## Qwen3 fine-tuning on SQuARE-Wikidata — contrastive recipe

Fine-tunes **`Qwen/Qwen3-Embedding-0.6B`** on the **SQuARE** benchmark
(result-set-grounded SPARQL query pairs with answer-set Jaccard supervision),
reusing the SparqlCE main-branch training recipe from
`notebooks/training/train_wikidata.ipynb`:

    loss      = InfoNCE(in-batch, tau=0.05) + 0.3 * MSE(cos(a,p), jaccard(a,p))
    optimizer = AdamW, cosine schedule with warmup, grad-clip 1.0
    selection = best val MRR (NOT mean cosine — that is biased toward collapse)
    splits    = paper's fixed split (splits_<kg>.json), seeded fallback otherwise

**What is revised vs. SparqlCE:**

| Component        | SparqlCE (CodeBERT + GATv2)                  | This notebook (Qwen3)                          |
|------------------|----------------------------------------------|------------------------------------------------|
| Input            | Query parsed into a graph (`SPARQLGraphBuilder`) | Raw SPARQL text, tokenized directly        |
| Encoder          | CodeBERT token emb + GATv2 + mean pool + proj | Qwen3-Embedding decoder, **last-token pooling** |
| Trainable params | Full model, differential LRs (1e-5 / 1e-4)    | **LoRA adapters** only, single LR 1e-4         |
| Precision        | fp32                                          | bf16 backbone + gradient checkpointing         |
| Augmentation     | Random variable renaming (graph-side)         | Same augmentation, reused on the text side     |

The loss, split handling, ranking-based validation, and checkpoint-selection
logic are carried over unchanged so the two models are directly comparable.

Run top-to-bottom. Verify the paths in the CONFIG cell first.

> **Data prerequisite:** `datasets/wikidata-triplets-dataset.json` is *not*
> tracked in this repo (see `datasets/README.md`). Place it there before
> running — the splits file `datasets/splits_wikidata.json` indexes into it
> (10,735 anchor groups: 8,588 train / 1,073 val / 1,074 test).


In [ ]:
# Dependencies (uncomment on a fresh environment)
# %pip install -q torch transformers>=4.51 peft accelerate tqdm


In [1]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm

# ============================================================================
# 1. CONFIG — verify these paths match your repo before running
# ============================================================================
KG = "wikidata"  # this notebook is the Wikidata run; see 04_* for DBpedia

PATHS = {
    "dbpedia": {
        "data":   "../../datasets/dbpedia-triplets-dataset.json",
        "splits": "../../datasets/splits_dbpedia.json",
    },
    "wikidata": {
        "data":   "../../datasets/wikidata-triplets-dataset.json",
        "splits": "../../datasets/splits_wikidata.json",
    },
}
DATA_PATH   = PATHS[KG]["data"]
SPLITS_PATH = PATHS[KG]["splits"]
CKPT_DIR    = f"../../checkpoints/qwen3_square_{KG}"

MODEL_NAME  = "Qwen/Qwen3-Embedding-0.6B"

# Standard contrastive fine-tuning recipe: in-batch InfoNCE, i.e.
# MultipleNegativesRankingLoss. SparqlCE-specific components (Jaccard
# calibration, graph encoder, variable-renaming augmentation) are
# DELIBERATELY EXCLUDED so this baseline credits only standard adaptation
# of a strong LLM embedder on SQuARE (see src-code/BASELINE_REDESIGN.md).
SEED         = 42
EPOCHS       = 5             # LoRA on a 0.6B model converges much faster than
                             # the from-scratch GNN (paper used 20 for SparqlCE)
TEMPERATURE  = 0.05          # paper: 0.05 (NOT 0.07)
LAMBDA_REG   = 0.0           # calibration OFF: the graded Jaccard-MSE term is
                             # SparqlCE's method, excluded from the baseline.
                             # (set to 0.3 only to reproduce a calibrated control)
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0

# Qwen3-specific settings
BATCH_SIZE   = 16            # per-step; in-batch negatives come from the batch
GRAD_ACCUM   = 2             # effective batch 32, matches the SparqlCE recipe
LR           = 1e-4          # single LR on LoRA params (backbone is frozen)
MAX_LEN      = 512           # tokens per SPARQL query
WARMUP_FRAC  = 0.05
AUGMENT_RENAME_PROB = 0.0    # no domain-specific augmentation in the standard baseline

LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]


/workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================================
# 2. Setup: repo import path, seeds, device
# ============================================================================
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Reuse the variable-renaming augmentation from the SparqlCE dataset module so
# both models see the same invariance signal during training.
from sparqlce.dataset import _random_rename_variables  # noqa: E402

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")


device = cuda


/workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: Could not load this library: /workspace/.conda/envs/sparql-env/lib/python3.10/site-packages/torch_spline_conv/_ve

### 3. Data

Same SQuARE JSON schema as `SemanticSPARQLDataset`
(`anchor_query` / `positive_queries` / `negative_queries`, each candidate
carrying its `answer_jaccard`), but the dataset below is **text-only**: no
graph builder, no PyG batching. In in-batch-negatives mode each item yields
`(anchor_text, positive_text, jaccard)` — negatives are the other batch items,
exactly as in the SparqlCE recipe.


In [3]:
class SquareTextDataset(Dataset):
    """Anchor-level SQuARE dataset over raw SPARQL text.

    Text-side revision of ``sparqlce.dataset.SemanticSPARQLDataset``
    (in-batch-negatives mode): identical JSON parsing and augmentation,
    minus the graph construction.
    """

    def __init__(self, datapath, augment_rename_prob=0.0, augment_seed=None):
        self.augment_rename_prob = float(augment_rename_prob)
        self._aug_rng = random.Random(augment_seed) if augment_seed is not None else random
        self.groups = []

        print(f"Loading dataset from {datapath}...")
        with open(datapath, "r", encoding="utf-8") as f:
            raw = json.load(f)

        for item in raw:
            anchor = item.get("anchor_query")
            if not anchor:
                continue
            pos_list = [
                (p["query"], float(p["answer_jaccard"]))
                for p in item.get("positive_queries", [])
                if p.get("query") is not None and p.get("answer_jaccard") is not None
            ]
            neg_list = [
                (n["query"], float(n["answer_jaccard"]))
                for n in item.get("negative_queries", [])
                if n.get("query") is not None and n.get("answer_jaccard") is not None
            ]
            if not pos_list or not neg_list:
                continue
            self.groups.append({"anchor": anchor, "positives": pos_list})

        print(f"Loaded {len(self.groups)} anchor groups.")

    def _maybe_augment(self, query):
        if self.augment_rename_prob > 0.0 and self._aug_rng.random() < self.augment_rename_prob:
            return _random_rename_variables(query, self._aug_rng)
        return query

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        g = self.groups[idx]
        pos_q, pos_j = random.choice(g["positives"])
        return self._maybe_augment(g["anchor"]), self._maybe_augment(pos_q), float(pos_j)


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")

def collate_pairs(batch):
    """Tokenize (anchor, positive, jaccard) triples into two padded batches."""
    anchors, positives, pos_js = zip(*batch)
    a_tok = tokenizer(list(anchors),   padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    p_tok = tokenizer(list(positives), padding=True, truncation=True,
                      max_length=MAX_LEN, return_tensors="pt")
    return a_tok, p_tok, torch.tensor(pos_js, dtype=torch.float32)


In [4]:
# ============================================================================
# 3b. Fixed split (paper) with safe random fallback — unchanged from SparqlCE
# ============================================================================
dataset = SquareTextDataset(
    DATA_PATH,
    augment_rename_prob=AUGMENT_RENAME_PROB,
    augment_seed=SEED,
)
# Validation/test must see un-augmented queries: share the file, not the flags.
eval_dataset = SquareTextDataset(DATA_PATH, augment_rename_prob=0.0)

if os.path.exists(SPLITS_PATH):
    with open(SPLITS_PATH, "r") as f:
        splits = json.load(f)
    train_idx = splits.get("train_idx", splits.get("train"))
    val_idx   = splits.get("val_idx",   splits.get("valid_idx", splits.get("val")))
    test_idx  = splits.get("test_idx",  splits.get("test"))
    assert train_idx and val_idx and test_idx, f"Could not parse splits from {SPLITS_PATH}"

    train_dataset = Subset(dataset,      train_idx)
    val_dataset   = Subset(eval_dataset, val_idx)
    test_dataset  = Subset(eval_dataset, test_idx)
    print(f"Using FIXED split from {SPLITS_PATH}")
else:
    print(f"WARNING: fixed split file not found at {SPLITS_PATH}")
    print("WARNING: falling back to seeded random_split — numbers will NOT match the paper.")
    total   = len(dataset)
    n_train = int(0.80 * total)
    n_val   = int(0.10 * total)
    n_test  = total - n_train - n_val
    gen = torch.Generator().manual_seed(SEED)
    train_split, val_split, test_split = random_split(
        range(total), [n_train, n_val, n_test], generator=gen)
    train_dataset = Subset(dataset,      list(train_split.indices))
    val_dataset   = Subset(eval_dataset, list(val_split.indices))
    test_dataset  = Subset(eval_dataset, list(test_split.indices))

print(f"train={len(train_dataset)}  val={len(val_dataset)}  test={len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_pairs, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_pairs)


Loading dataset from ../../datasets/wikidata-triplets-dataset.json...
Loaded 10735 anchor groups.
Loading dataset from ../../datasets/wikidata-triplets-dataset.json...
Loaded 10735 anchor groups.
Using FIXED split from ../../datasets/splits_wikidata.json
train=8588  val=1073  test=1074


### 4. Model: Qwen3-Embedding + LoRA

Qwen3-Embedding is a decoder-only model: the query embedding is the hidden
state of the **last non-padding token** (the pooling recommended on the model
card), L2-normalised — this replaces SparqlCE's GATv2 + mean-pool + projection
head. Only LoRA adapters are trained; the 0.6B backbone stays frozen in bf16.


In [5]:
def last_token_pool(last_hidden_states, attention_mask):
    """Last-token pooling as on the Qwen3-Embedding model card.
    Handles both left- and right-padded batches."""
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    idx = torch.arange(last_hidden_states.shape[0], device=last_hidden_states.device)
    return last_hidden_states[idx, seq_lens]


class Qwen3SparqlEmbedder(nn.Module):
    """Qwen3-Embedding backbone + LoRA, producing L2-normalised embeddings.

    Same output contract as ``HybridSPARQLEncoder.forward`` (unit-norm float
    embeddings), so the SparqlCE loss and evaluation code apply verbatim.
    """

    def __init__(self, model_name, lora_cfg=None, adapter_path=None):
        super().__init__()
        base = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        base.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False})
        base.enable_input_require_grads()  # needed for checkpointing + frozen base
        if adapter_path is not None:
            self.backbone = PeftModel.from_pretrained(base, adapter_path)
        else:
            self.backbone = get_peft_model(base, lora_cfg)

    def forward(self, tok):
        out = self.backbone(input_ids=tok["input_ids"],
                            attention_mask=tok["attention_mask"])
        emb = last_token_pool(out.last_hidden_state, tok["attention_mask"])
        return F.normalize(emb.float(), p=2, dim=1)


lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias="none",
    task_type="FEATURE_EXTRACTION",
)

model = Qwen3SparqlEmbedder(MODEL_NAME, lora_cfg=lora_cfg).to(device)
model.backbone.print_trainable_parameters()

# Single LR on the LoRA params replaces SparqlCE's differential-LR scheme:
# there is no random-init GNN here, and the backbone is frozen.
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR, weight_decay=WEIGHT_DECAY,
)
steps_per_epoch = len(train_loader) // GRAD_ACCUM
total_steps     = steps_per_epoch * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRAC * total_steps),
    num_training_steps=total_steps,
)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 966.98it/s, Materializing param=norm.weight]                               


trainable params: 10,092,544 || all params: 605,869,056 || trainable%: 1.6658


In [6]:
# ============================================================================
# 5. Loss — standard in-batch InfoNCE (== MultipleNegativesRankingLoss).
# No Jaccard calibration: that graded-similarity term is SparqlCE's contribution.
# ============================================================================
def infonce_inbatch(a_emb, p_emb, temperature):
    """In-batch InfoNCE. Matches the paper / main branch."""
    a = F.normalize(a_emb, dim=1)
    p = F.normalize(p_emb, dim=1)
    logits = a @ p.T / temperature
    labels = torch.arange(a.size(0), device=a.device)
    return F.cross_entropy(logits, labels)


In [7]:
# ============================================================================
# 6. Validation: rank each anchor's true positive against ALL positives in
# the split. Identical to SparqlCE — selects the embedding that retrieves
# correctly, not the one that collapses everything to a single point.
# ============================================================================
@torch.no_grad()
def evaluate_ranking(model, loader, device):
    model.eval()
    anchors, positives = [], []
    for a_tok, p_tok, _ in loader:
        a_tok = {k: v.to(device) for k, v in a_tok.items()}
        p_tok = {k: v.to(device) for k, v in p_tok.items()}
        anchors.append(model(a_tok).cpu())
        positives.append(model(p_tok).cpu())
    A = torch.cat(anchors)      # [N, d]
    P = torch.cat(positives)    # [N, d]
    sims  = A @ P.T             # [N, N]; diagonal is the true positive
    diag  = sims.diag().unsqueeze(1)
    ranks = (sims >= diag).sum(dim=1).float()   # 1-indexed rank of true positive
    return {
        "mrr":    (1.0 / ranks).mean().item(),
        "hits@1": (ranks == 1).float().mean().item(),
        "hits@5": (ranks <= 5).float().mean().item(),
    }


In [8]:
# ============================================================================
# 7. Training loop — standard contrastive fine-tuning (gradient accumulation)
# ============================================================================
os.makedirs(CKPT_DIR, exist_ok=True)
best_mrr  = -1.0
best_path = os.path.join(CKPT_DIR, "best_adapter")
history   = []

print(f"Training Qwen3 on SQuARE-{KG} for {EPOCHS} epochs "
      f"(effective batch {BATCH_SIZE * GRAD_ACCUM})")
print("loss = InfoNCE(tau=%s)%s" % (TEMPERATURE,
      f" + {LAMBDA_REG}*MSE(cos,jaccard)" if LAMBDA_REG > 0
      else " — standard in-batch InfoNCE / MNR, no calibration"))

for epoch in range(1, EPOCHS + 1):
    model.train()
    sum_loss, sum_ce, sum_reg, n_steps = 0.0, 0.0, 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")

    for step, (a_tok, p_tok, pos_js) in enumerate(pbar, start=1):
        a_tok  = {k: v.to(device) for k, v in a_tok.items()}
        p_tok  = {k: v.to(device) for k, v in p_tok.items()}
        pos_js = pos_js.to(device)

        a_emb = model(a_tok)
        p_emb = model(p_tok)

        # InfoNCE on the batch (in-batch negatives) == MultipleNegativesRankingLoss
        loss_ce = infonce_inbatch(a_emb, p_emb, TEMPERATURE)

        # Jaccard calibration MSE — SparqlCE's graded objective. OFF for the
        # standard baseline (LAMBDA_REG=0); re-enable only for a calibrated control.
        if LAMBDA_REG > 0:
            cos = (a_emb * p_emb).sum(dim=1)
            loss_reg = (cos - pos_js).pow(2).mean()
            loss = loss_ce + LAMBDA_REG * loss_reg
        else:
            loss_reg = torch.zeros((), device=a_emb.device)
            loss = loss_ce
        (loss / GRAD_ACCUM).backward()

        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad), max_norm=GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        sum_loss += loss.item(); sum_ce += loss_ce.item(); sum_reg += loss_reg.item()
        n_steps  += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         ce=f"{loss_ce.item():.4f}",
                         reg=f"{loss_reg.item():.4f}")

    val_metrics = evaluate_ranking(model, val_loader, device)
    history.append({"epoch": epoch,
                    "train_loss": sum_loss / n_steps,
                    **{f"val_{k}": v for k, v in val_metrics.items()}})
    print(f"Epoch {epoch:02d} | loss {sum_loss/n_steps:.4f} "
          f"(ce {sum_ce/n_steps:.4f}, reg {sum_reg/n_steps:.4f}) | "
          f"val MRR {val_metrics['mrr']:.4f} "
          f"H@1 {val_metrics['hits@1']:.4f} H@5 {val_metrics['hits@5']:.4f}")

    # Selection by val MRR, as in the SparqlCE recipe. Only the LoRA adapter
    # is saved (~a few MB) instead of a full state dict.
    if val_metrics["mrr"] > best_mrr:
        best_mrr = val_metrics["mrr"]
        model.backbone.save_pretrained(best_path)
        tokenizer.save_pretrained(best_path)
        print(f"  ✓ new best val MRR {best_mrr:.4f} — adapter saved to {best_path}")


Training Qwen3 on SQuARE-wikidata for 5 epochs (effective batch 32)
loss = InfoNCE(tau=0.05) + 0.3 * MSE(cos, jaccard)


Epoch 01/5: 100%|██████████| 536/536 [10:24<00:00,  1.16s/it, ce=1.2390, loss=1.2414, reg=0.0081]


Epoch 01 | loss 0.7806 (ce 0.7762, reg 0.0147) | val MRR 0.4099 H@1 0.3383 H@5 0.4697
  ✓ new best val MRR 0.4099 — adapter saved to ../../checkpoints/qwen3_square_wikidata/best_adapter


Epoch 02/5: 100%|██████████| 536/536 [10:22<00:00,  1.16s/it, ce=0.2775, loss=0.2817, reg=0.0141]


Epoch 02 | loss 0.6749 (ce 0.6720, reg 0.0097) | val MRR 0.4190 H@1 0.3514 H@5 0.4772
  ✓ new best val MRR 0.4190 — adapter saved to ../../checkpoints/qwen3_square_wikidata/best_adapter


...

Epoch 03 | loss 0.5869 (ce 0.5838, reg 0.0103) | val MRR 0.4174 H@1 0.3420 H@5 0.4800


Epoch 04/5: 100%|██████████| 536/536 [10:22<00:00,  1.16s/it, ce=0.3957, loss=0.3979, reg=0.0073]


Epoch 04 | loss 0.5624 (ce 0.5592, reg 0.0104) | val MRR 0.4281 H@1 0.3523 H@5 0.4902
  ✓ new best val MRR 0.4281 — adapter saved to ../../checkpoints/qwen3_square_wikidata/best_adapter


Epoch 05/5: 100%|██████████| 536/536 [10:23<00:00,  1.16s/it, ce=0.3145, loss=0.3174, reg=0.0098]


Epoch 05 | loss 0.5369 (ce 0.5337, reg 0.0108) | val MRR 0.4245 H@1 0.3430 H@5 0.5014


In [9]:
# ============================================================================
# 8. Sanity check on the training-file test split (pair-ranking metric)
#
# NOTE: this is only the quick diagonal-ranking check used during training.
# The numbers reported in the paper's RQ1 tables come from Section 9 below,
# which follows the exact protocol of notebooks/evaluation-RQ1/evaluate_rq1.py.
# ============================================================================
print(f"Training complete. Best val MRR: {best_mrr:.4f}")
print("Reloading best adapter and evaluating on the test split...")

del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

best_model = Qwen3SparqlEmbedder(MODEL_NAME, adapter_path=best_path).to(device)
test_metrics = evaluate_ranking(best_model, test_loader, device)

print(f"\nSQuARE-{KG} test sanity check (fine-tuned Qwen3-Embedding-0.6B):")
for k, v in test_metrics.items():
    print(f"  {k:>7}: {v:.4f}")

with open(os.path.join(CKPT_DIR, "test_metrics.json"), "w") as f:
    json.dump({"kg": KG, "model": MODEL_NAME, "best_val_mrr": best_mrr,
               "test": test_metrics, "history": history}, f, indent=2)
print(f"\nMetrics written to {os.path.join(CKPT_DIR, 'test_metrics.json')}")


Training complete. Best val MRR: 0.4281
Reloading best adapter and evaluating on the test split...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1005.31it/s, Materializing param=norm.weight]                              



SQuARE-wikidata test sanity check (fine-tuned Qwen3-Embedding-0.6B):
      mrr: 0.4335
   hits@1: 0.3538
   hits@5: 0.5158

Metrics written to ../../checkpoints/qwen3_square_wikidata/test_metrics.json


### 9. RQ1 evaluation on the held-out test set (paper tables)

This is the evaluation that fills the **"Qwen3-Emb-0.6B (fine-tuned)"** row in
the paper's `tables/tab-RQ1-Wikidata.tex`.
It follows the **exact** protocol of `notebooks/evaluation-RQ1/evaluate_rq1.py`,
the same script used for all baselines and SparqlCE:

- **Data:** `datasets/wikidata_testset_RQ1.json` (186 anchors, 2227 corpus
  queries) — *not* the training-file split used above for model selection.
- **Retrieval:** each anchor is ranked against the **global corpus** (all unique
  anchor/positive/negative queries in the test set), anchor self-match excluded.
- **Metrics:** MRR, Hits, nDCG (ranking) and P, R, F1 (set overlap) at
  k = 3, 5, 10, 20, averaged over anchors with ≥1 annotated positive.

The only model-specific part is the encoder: queries are embedded with the
fine-tuned Qwen3 model (best-val-MRR LoRA adapter, last-token pooling,
L2-normalised), mirroring `build_emb_map_sentence_transformer` in the script.


In [10]:
# ============================================================================
# 9a. Load the RQ1 test set and build corpus / eval items / relevance maps
#     — exactly as notebooks/evaluation-RQ1/{DBpedia,Wikidata}/06_eval_*.ipynb
# ============================================================================
_RQ1_DIR = str(Path(PROJECT_ROOT) / "notebooks" / "evaluation-RQ1")
if _RQ1_DIR not in sys.path:
    sys.path.insert(0, _RQ1_DIR)

from evaluate_rq1 import (  # noqa: E402
    load_triplets, build_global_corpus, build_eval_items,
    build_binary_relevance_maps, build_sorted_pools_global,
    avg_prf_at_k_annotated, avg_mrr_at_k_annotated, avg_hits_at_k_annotated,
    avg_ndcg_binary, report,
)

RQ1_TEST_PATH = f"../../datasets/{KG}_testset_RQ1.json"
TOPKS = [3, 5, 10, 20]   # same k values as the paper tables

rq1_data = load_triplets(RQ1_TEST_PATH)
rq1_corpus, rq1_seen_q = build_global_corpus(rq1_data)
rq1_eval_items = build_eval_items(rq1_data, rq1_corpus, rq1_seen_q)
rq1_relevance_maps = build_binary_relevance_maps(rq1_eval_items, rq1_corpus)

# For Wikidata this should print: anchors=186, corpus=2227 (as in the paper).
print(f"RQ1 test set loaded from {RQ1_TEST_PATH}")
print(f"Evaluation ready -> anchors: {len(rq1_eval_items)} | corpus: {len(rq1_corpus)}")


RQ1 test set loaded from ../../datasets/wikidata_testset_RQ1.json
Evaluation ready -> anchors: 186 | corpus: 2229


In [11]:
# ============================================================================
# 9b. Encode the RQ1 corpus with the fine-tuned model and rank
# ============================================================================
@torch.no_grad()
def build_emb_map_qwen3(model, queries, batch_size=16, max_length=MAX_LEN):
    """Encode raw SPARQL strings with the fine-tuned Qwen3 embedder.

    Uses the same tokenizer settings and last-token pooling as training;
    the model already returns L2-normalised float embeddings, so cosine
    similarity is a plain dot product (as evaluate_rq1 assumes).
    """
    model.eval()
    emb_map = {}
    for i in tqdm(range(0, len(queries), batch_size), desc="Encoding RQ1 corpus"):
        batch = queries[i:i + batch_size]
        tok = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        tok = {k: v.to(device) for k, v in tok.items()}
        embs = model(tok).cpu().numpy()
        for q, e in zip(batch, embs):
            emb_map[q] = e
    return emb_map


# `best_model` holds the best-val-MRR adapter reloaded in Section 8. If you
# are re-running only this section in a fresh kernel, uncomment:
# best_model = Qwen3SparqlEmbedder(MODEL_NAME, adapter_path=best_path).to(device)

rq1_emb_map = build_emb_map_qwen3(best_model, rq1_corpus, batch_size=BATCH_SIZE)
rq1_sorted_pools = build_sorted_pools_global(rq1_emb_map, rq1_eval_items, rq1_corpus)

report(f"Qwen3-Emb-0.6B (fine-tuned) — RQ1 {KG}",
       rq1_sorted_pools, rq1_eval_items, rq1_relevance_maps, TOPKS)


Encoding RQ1 corpus: 100%|██████████| 140/140 [00:17<00:00,  8.00it/s]



Qwen3-Emb-0.6B (fine-tuned) — RQ1 wikidata
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1452  0.0323  0.0489  0.1541  0.2366    0.1452  0.1335
@5     0.1656  0.0599  0.0807  0.1727  0.3172    0.1656  0.1522
@10    0.1817  0.1270  0.1361  0.1927  0.4677    0.1817  0.1774
@20    0.1788  0.2458  0.1876  0.2009  0.5860    0.1788  0.2150


In [12]:
# ============================================================================
# 9c. Collect metrics per k, emit LaTeX rows for the paper tables, save JSON
# ============================================================================
rq1_metrics = {}
for k in TOPKS:
    p, r, f1 = avg_prf_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k)
    rq1_metrics[k] = {
        "MRR":  avg_mrr_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k),
        "Hits": avg_hits_at_k_annotated(rq1_sorted_pools, rq1_eval_items, k),
        "nDCG": avg_ndcg_binary(rq1_sorted_pools, rq1_relevance_maps, k),
        "P": p, "R": r, "F1": f1,
    }

# Rows in the exact column order of tables/tab-RQ1-<KG>.tex
# (per k: MRR, Hits, nDCG for the ranking subtable; P, R, F1 for set-overlap).
rank_row = " & ".join(f"{rq1_metrics[k][m]:.4f}"
                      for k in TOPKS for m in ("MRR", "Hits", "nDCG"))
ovlp_row = " & ".join(f"{rq1_metrics[k][m]:.4f}"
                      for k in TOPKS for m in ("P", "R", "F1"))

print("LaTeX row — ranking-metrics subtable:")
print(f"Qwen3-Emb-0.6B (fine-tuned)~\\cite{{qwen2025qwen3}} & {rank_row} \\\\")
print("\nLaTeX row — set-overlap subtable:")
print(f"Qwen3-Emb-0.6B (fine-tuned) & {ovlp_row} \\\\")

rq1_out_path = os.path.join(CKPT_DIR, f"rq1_test_metrics_{KG}.json")
with open(rq1_out_path, "w") as f:
    json.dump({
        "kg": KG,
        "model": f"{MODEL_NAME} (fine-tuned, best adapter)",
        "protocol": "notebooks/evaluation-RQ1/evaluate_rq1.py",
        "testset": RQ1_TEST_PATH,
        "anchors": len(rq1_eval_items),
        "corpus_size": len(rq1_corpus),
        "topks": TOPKS,
        "metrics": {str(k): rq1_metrics[k] for k in TOPKS},
    }, f, indent=2)
print(f"\nRQ1 metrics written to {rq1_out_path}")


LaTeX row — ranking-metrics subtable:
Qwen3-Emb-0.6B (fine-tuned)~\cite{qwen2025qwen3} & 0.1541 & 0.2366 & 0.1335 & 0.1727 & 0.3172 & 0.1522 & 0.1927 & 0.4677 & 0.1774 & 0.2009 & 0.5860 & 0.2150 \\

LaTeX row — set-overlap subtable:
Qwen3-Emb-0.6B (fine-tuned) & 0.1452 & 0.0323 & 0.0489 & 0.1656 & 0.0599 & 0.0807 & 0.1817 & 0.1270 & 0.1361 & 0.1788 & 0.2458 & 0.1876 \\

RQ1 metrics written to ../../checkpoints/qwen3_square_wikidata/rq1_test_metrics_wikidata.json


### Notes

- **Paper numbers come from Section 9.** The RQ1 tables in the paper are
  computed on `datasets/{kg}_testset_RQ1.json` with the shared
  `evaluate_rq1.py` protocol; Section 8's pair-ranking numbers are only a
  training sanity check and are **not** comparable to the tables.
- **Comparability.** Loss, temperature, calibration weight, fixed splits, and
  the MRR-based selection are identical to the SparqlCE training notebook, and
  the RQ1 evaluation reuses the exact same script as all baselines — so the
  Section 9 output slots directly into the paper's tables as the
  *fine-tuned* Qwen3 row (vs. the zero-shot Qwen3 baseline).
- **Zero-shot reference.** To reproduce the zero-shot Qwen3 row with the same
  pipeline, skip Section 7 and run Section 9 on a fresh
  `Qwen3SparqlEmbedder(MODEL_NAME, lora_cfg=lora_cfg)` — the untrained LoRA
  is an identity at init (`lora_B = 0`), so it reproduces the base model.
- **DBpedia.** This notebook is preset to Wikidata; the DBpedia run lives in
  `04_qwen3_contrastive_finetune_square.ipynb`. Only `KG` and the markdown
  differ — the recipe is identical.
- **Missing training file.** `datasets/wikidata-triplets-dataset.json` must be
  placed manually (it is not tracked in the repo); Section 3b will raise a
  clear error if it is absent. The RQ1 test set
  (`datasets/wikidata_testset_RQ1.json`) *is* in the repo.
- **Memory.** ~0.6B params in bf16 + LoRA + checkpointing fits comfortably on a
  single 24 GB GPU at batch 16 / 512 tokens. Reduce `BATCH_SIZE` and raise
  `GRAD_ACCUM` proportionally if you hit OOM — the loss uses in-batch
  negatives, so the *per-step* batch (not effective batch) sets negative count.
